# Tecan Carousel: a plate hotel on the transfer station

```{device-card} tecan-carousel
```

| Property | Value |
|---|---|
| Transport | Operator-selected serial port, 9600 baud, 8E1, RTS/CTS |
| Protocol | StoreX remote operation: `CR` opens the link, one `<payload>\r\n` reply per command |
| Scope | Status reads, positioning, and plate handling between cartridge levels and the transfer station |
| Safety | Only `setup()` is passive; positioning and every handling operation move the machine |

**Untested integrated driver.** The unit is a **Tecan Carousel LPT220 EVO Microplate Hotel**, sold by
Tecan and manufactured by Liconic. The status reads, `ST 1905` export, `ST 1800` soft reset
and `ST 1906` placement have been observed working on this unit; the remaining handling commands
follow the manufacturer's command reference but have not been exercised here. Confirm every step
with the operator until they have.


In [ ]:
# **HARDWARE: do not execute without per-run operator approval.**
from pylabrobot.tecan import TecanCarousel

## How it talks

`setup()` opens PLR's serial transport, clears the input buffer, sends `CR\r` and requires
`CC\r\n`. Every later command is answered by exactly one CRLF-terminated frame. Handling commands
(`ST …`) are acknowledged with `OK` and complete when the ready bit returns to `1`; the driver polls
`RD 1915` and `RD 1814` instead of sleeping a fixed time.

Nothing is retried. A timeout, a malformed frame, an I/O failure or an in-flight cancellation faults
the session, which then requires `stop()` and `setup()`. A controller rejection (`E0`–`E5`) is
reported as `CarouselRejected` but does not fault the session, because the link is still in sync.
Closing the serial port is not an emergency stop.


## Physical setup

Have an operator confirm the serial cable and port, the installed cartridges, and the clear motion
path. There is deliberately no default port. The two geometry arguments are the installation's
business: the default of ten cartridges of 22 levels is the manual's maximum for standard cassettes;
half-height and deep-well cassettes hold fewer plates.

Levels are controller addresses and start at the bottom of a cartridge. They are not the top-down
slot numbers used in lab notes, and this mapping has not been verified for this unit.


In [ ]:
# **HARDWARE: do not execute without per-run operator approval.**
port = "REPLACE_WITH_OPERATOR_SELECTED_PORT"
carousel = TecanCarousel(port=port, number_of_cartridges=10, levels_per_cartridge=22)

## Connect, which does not initialize

`setup()` opens the link and completes the handshake only. It does not activate, home or move the
machine, and a ready controller is not proof of a safe mechanism.


In [ ]:
# **HARDWARE: do not execute without per-run operator approval.**
await carousel.setup()

## Read the whole state

One serialized snapshot of the ready, error, active and plate-ready flags, both plate sensors, the
selected cartridge and level, and the raw `DM200` process word.

`process_status` is deliberately not decoded: the documented error table does not describe the
values this unit reports, so no meaning is invented for them. A latched error carries the same raw
word on the exception's `code` attribute.


In [ ]:
# **HARDWARE: do not execute without per-run operator approval.**
status = await carousel.get_status()
status

## The two plate sensors

`RD 1812` reads the shovel and `RD 1813` the transfer station. These are presence sensors, not proof
that a plate arrived where you expected: after a completed handling action the driver compares them
with the expected result and raises rather than reporting success it cannot see.


In [ ]:
# **HARDWARE: do not execute without per-run operator approval.**
print("shovel:", await carousel.is_plate_on_shovel())
print("transfer station:", await carousel.is_plate_at_transfer_station())

## Initialize

`initialize()` activates and homes the handling with `ST 1801`. It reads the activation flag first
and does nothing when the handling is already active, so it is safe to call after every connection.

**MOTION:** the carousel rotates and the lift homes.


In [ ]:
# **HARDWARE: do not execute without per-run operator approval.** MOTION.
await carousel.initialize()

## Retrieve a plate to the transfer station

`export_plate(cartridge, level)` rotates and lifts, picks the plate with the shovel and places it on
the transfer station (`WR DM0`, `WR DM5`, `ST 1905`).

The method refuses to start unless the controller is ready with no error, the shovel is empty and the
transfer station is empty, and it raises if the transfer station does not sense the plate afterwards.
An empty source slot stops the operation with the controller's own PlateTrace error instead of
delivering nothing quietly.


In [ ]:
# **HARDWARE: do not execute without per-run operator approval.** MOTION.
await carousel.export_plate(cartridge=9, level=22)

## Return a plate into the carousel

`import_plate(cartridge, level)` takes the plate from the transfer station into a cartridge level
(`ST 1904`). It requires a plate on the transfer station and an empty shovel.


In [ ]:
# **HARDWARE: do not execute without per-run operator approval.** MOTION.
await carousel.import_plate(cartridge=9, level=22)

## Release a plate that is stuck on the shovel

`put_plate_on_transfer_station()` sends `ST 1906`, which places the plate already on the shovel onto
the transfer station. This is the documented way out of a half-finished access; it does not start a
new cartridge access.

`soft_reset()` (`ST 1800`) is the companion recovery: it releases the current access without moving
the plate, after which the plate's physical position still needs the operator's eyes.


In [ ]:
# **HARDWARE: do not execute without per-run operator approval.** MOTION.
await carousel.soft_reset()
await carousel.put_plate_on_transfer_station(cartridge=9, level=22)

## Move a plate inside the carousel

`move_plate()` picks from one level and places into another (`ST 1908` + `ST 1909`), verifying the
pick before it rotates to the destination, so a failed pick can never become a placement into an
unknown position. The transfer station is not used and must be empty.


In [ ]:
# **HARDWARE: do not execute without per-run operator approval.** MOTION.
await carousel.move_plate(cartridge=2, level=17, to_cartridge=2, to_level=15)

## Clear a latched handling error

`reset()` sends `ST 1900`: it clears the error flag and `DM200`, re-homes the mechanism and leaves
the handling inactive, so call `initialize()` afterwards. It is the documented way to clear a
latched error, not a retry of whatever failed.


In [ ]:
# **HARDWARE: do not execute without per-run operator approval.** MOTION.
await carousel.reset()
await carousel.initialize()

## Close the connection

`stop()` sends `CQ` to end the controller conversation and closes the port. It does not stop
motion, deactivate the controller or make manual loading safe. Always close in a `finally` block in
application code.


In [ ]:
# **HARDWARE: do not execute without per-run operator approval.**
await carousel.stop()